In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
# from dataclasses import dataclass
from typing import List, Dict, Any

import xml.etree.ElementTree as ET

In [2]:

BASE_DIR = Path.cwd().parents[0]


load_dotenv(BASE_DIR / "creds" / ".env")




True

In [3]:
def xml_to_dict(element) -> Dict[str, Any]:
    """Convert XML element to clean dictionary."""
    result = {}
    
    # Add attributes if they exist
    if element.attrib:
        result['_attributes'] = element.attrib
    
    # Add text content
    if element.text and element.text.strip():
        result['_text'] = element.text.strip()
    
    # Process child elements
    for child in element:
        child_data = xml_to_dict(child)
        
        # Handle multiple elements with same tag
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_data)
        else:
            # Single element - just store the data
            result[child.tag] = child_data if child_data else (child.text or None)
    
    return result if result else None


def parse_vehicle_search(xml_string: str) -> Dict:
    """Parse vehicle search response into clean data."""
    root = ET.fromstring(xml_string)
    
    # Extract header info
    header = root.find('Header')
    response_data = {
        'status': header.findtext('Status') if header is not None else None,
        'status_code': header.findtext('StatusCode') if header is not None else None,
        'vehicles': []
    }
    
    # Extract vehicle data
    for vehicle_elem in root.findall('.//VehicleSearchItem'):
        # Extract all links
        links = []
        links_container = vehicle_elem.find('Links')
        if links_container is not None:
            for link in links_container.findall('Link'):
                links.append({
                    'href': link.findtext('Href'),
                    'rel': link.findtext('Rel')
                })
        
        vehicle = {
            'base_vehicle_id': vehicle_elem.findtext('BaseVehicleID'),
            'make_name': vehicle_elem.findtext('MakeName'),
            'model_name': vehicle_elem.findtext('ModelName'),
            'sub_model_name': vehicle_elem.findtext('SubModelName'),
            'year': vehicle_elem.findtext('Year'),
            'engine_description': vehicle_elem.findtext('EngineDescription'),
            'vehicle_id': vehicle_elem.findtext('VehicleID'),
            'is_active': vehicle_elem.findtext('VehicleIsActive') == 'true',
            'links': links
        }
        response_data['vehicles'].append(vehicle)
    
    return response_data

<h2>Set Up</h2>
<p>
Run this first to set up the required functions:
</p>

In [4]:
from datetime import datetime, timezone
from urllib.parse import urlparse
import urllib
import hmac
import hashlib
import requests
import base64

C_PUBLIC_KEY = os.getenv("C_PUBLIC_KEY")
C_PRIVATE_KEY = os.getenv("C_PRIVATE_KEY")


if (not C_PUBLIC_KEY) | (not C_PRIVATE_KEY):
    raise EnvironmentError("API keys not set") 

def GenerateSharedAuth(d: datetime, public_key: str, private_key: str, uri: str, http_verb: str) -> bytes:
    utc_time = d.astimezone(timezone.utc)
    epoch_time = int(utc_time.timestamp())
    relative_url = urlparse(uri).path
    encoded_url = urllib.parse.quote(relative_url)

    plain_sig = public_key + chr(10) + http_verb + chr(10) + str(epoch_time) + chr(10) + encoded_url
    key = private_key.encode('ascii')
    byte_sig = plain_sig.encode('ascii')

    signature = hmac.new(key, byte_sig, hashlib.sha256).digest()
    return signature

def GetResponse(uri: str) -> str:
    today = datetime.now()
    auth = GenerateSharedAuth(today, C_PUBLIC_KEY, C_PRIVATE_KEY, uri, "GET")
    headers = {"Authorization": "Shared " + C_PUBLIC_KEY + ":" + base64.b64encode(auth).decode('ascii'), 
               "Date": today.astimezone(timezone.utc).strftime("%a, %d %b %Y %H:%M:%S GMT"),
               "Host": "api.motor.com"}

    response = requests.get("https://api.motor.com" + uri, headers=headers)
    return response.text

In [5]:


def extract_keywords_from_xml(xml_string: str, debug: bool = False) -> Dict:
    """
    Extract status, status_code, ApplicationID, and DisplayName from any XML response.
    Returns application_ids as a list of dicts with id and display_name.
    Handles nested ApplicationIDs and XML namespaces.
    Strips quotes from extracted values.
    """
    root = ET.fromstring(xml_string)

    # Strip namespace from tag names for easier searching
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag

    def strip_quotes(value):
        """Remove surrounding quotes from string values."""
        if value and isinstance(value, str):
            return value.strip('\'"')
        return value

    # Extract status and status_code
    result = {
        'status': None,
        'status_code': None,
        'application_ids': []  # List of dicts with id and display_name
    }

    # Search for ApplicationID/DisplayName pairs within same parent
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = strip_quotes(elem.text)
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = strip_quotes(elem.text)
        elif tag == 'EstimatedWorkTimeApplicationSummary' or tag == 'ApplicationSummary':
            # Extract paired ApplicationID and DisplayName from same parent
            app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
            display_name_elem = next((e for e in elem if strip_ns(e.tag) == 'DisplayName'), None)

            if app_id_elem is not None and app_id_elem.text:
                app_id = strip_quotes(app_id_elem.text)
                display_name = strip_quotes(display_name_elem.text) if display_name_elem is not None and display_name_elem.text else None
                
                # Append as dict to list
                result['application_ids'].append({
                    'id': app_id,
                    'display_name': display_name if display_name else 'Unknown'
                })

    if debug:
        print(f"Found {len(result['application_ids'])} ApplicationID(s)")

    return result


def extract_application_id(xml_string: str) -> str:
    """
    Extract ApplicationID from any XML response.
    Returns the first ApplicationID found.
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    # Find first ApplicationID element
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'ApplicationID' and elem.text:
            return elem.text
    
    return None


def extract_all_part_application_ids(xml_string: str, debug: bool = False) -> List[str]:
    """
    Extract ALL part ApplicationIDs from a parts-summary response.
    Returns a list of all ApplicationID values found within PartApplicationSummary/PartApp containers.
    """
    root = ET.fromstring(xml_string)

    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag

    ids = []

    # Look for ApplicationID within PartApplicationSummary/PartApp containers
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'PartApplicationSummary' or tag == 'PartApp':
            # Look for ApplicationID as direct child
            for child in elem:
                child_tag = strip_ns(child.tag)
                if child_tag == 'ApplicationID' and child.text:
                    ids.append(child.text)
                    if debug:
                        print(f"  Found ApplicationID: {child.text}")
                    break  # Only take first ApplicationID per container

    if debug:
        print(f"Total ApplicationIDs found: {len(ids)}")

    return ids


def extract_estimated_work_time(xml_string: str) -> Dict:
    """
    Extract EstimatedWorkTime details from XML response.
    Returns status, status_code, and a details dict with labor time and skill information.
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Helper to get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    result = {
        'status': None,
        'status_code': None,
        'details': {}
    }
    
    # Extract status and status_code
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = elem.text
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = elem.text
    
    # Find EstimatedWorkTime element
    work_time_elem = next((e for e in root.iter() if strip_ns(e.tag) == 'EstimatedWorkTime'), None)
    
    if work_time_elem is not None:
        # Extract Job Description from Notes > Note > Text
        job_description = None
        notes_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'Notes'), None)
        if notes_elem is not None:
            note_elem = next((e for e in notes_elem if strip_ns(e.tag) == 'Note'), None)
            if note_elem is not None:
                job_description = get_text(note_elem, 'Text')
        
        # Extract RequiredSkill > Description
        required_skill = None
        skill_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'RequiredSkill'), None)
        if skill_elem is not None:
            required_skill = get_text(skill_elem, 'Description')
        
        # Build details dict
        result['details'] = {
            'job_description': job_description,
            'additional_labor_time': get_text(work_time_elem, 'AdditionalLaborTime'),
            'additional_warranty_labor_time': get_text(work_time_elem, 'AdditionalWarrantyLaborTime'),
            'all_labor_time': get_text(work_time_elem, 'AllLaborTime'),
            'all_warranty_labor_time': get_text(work_time_elem, 'AllWarrantyLaborTime'),
            'base_labor_time': get_text(work_time_elem, 'BaseLaborTime'),
            'base_warranty_labor_time': get_text(work_time_elem, 'BaseWarrantyLaborTime'),
            'labor_time_interval': get_text(work_time_elem, 'LaborTimeInterval'),
            'required_skill': required_skill,
            'service_type': get_text(work_time_elem, 'ServiceType'),
            'base_labor_time_average': get_text(work_time_elem, 'BaseLaborTimeAverage'),
            'is_active': get_text(work_time_elem, 'IsActive'),
            'type': get_text(work_time_elem, 'Type')
        }
    
    return result


def extract_part_details(xml_string: str) -> Dict:
    """
    Extract part details from XML response.
    Returns status, status_code, and a details dict with part information.
    Handles nested structure: Part > PricingFamilies > PartPricingFamily > Pricing > PartPricing
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Helper to get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    result = {
        'status': None,
        'status_code': None,
        'details': {}
    }
    
    # Extract status and status_code
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = elem.text
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = elem.text
    
    # Find Part element
    part_elem = next((e for e in root.iter() if strip_ns(e.tag) == 'Part'), None)
    
    if part_elem is not None:
        part_number = get_text(part_elem, 'PartNumber')
        
        # Navigate to PartPricingFamily
        country_name = None
        manufacturer_name = None
        effective_date = None
        is_current = None
        oepr_part_number = None
        motor_part_number = None
        net_core_price = None
        category_name = None
        part_terminology_name = None
        price = None
        return_old_part = None
        
        pricing_families = next((e for e in part_elem if strip_ns(e.tag) == 'PricingFamilies'), None)
        if pricing_families is not None:
            part_pricing_family = next((e for e in pricing_families if strip_ns(e.tag) == 'PartPricingFamily'), None)
            if part_pricing_family is not None:
                # Extract CountryInfo > Name
                country_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'CountryInfo'), None)
                if country_elem is not None:
                    country_name = get_text(country_elem, 'Name')
                
                # Extract ManufacturerInfo > Name
                manufacturer_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'ManufacturerInfo'), None)
                if manufacturer_elem is not None:
                    manufacturer_name = get_text(manufacturer_elem, 'Name')
                
                # Navigate to Pricing > PartPricing
                pricing_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'Pricing'), None)
                if pricing_elem is not None:
                    part_pricing = next((e for e in pricing_elem if strip_ns(e.tag) == 'PartPricing'), None)
                    if part_pricing is not None:
                        effective_date = get_text(part_pricing, 'EffectiveDate')
                        is_current = get_text(part_pricing, 'IsCurrent')
                        oepr_part_number = get_text(part_pricing, 'OEPRPartNumber')
                        motor_part_number = get_text(part_pricing, 'MOTORPartNumber')
                        net_core_price = get_text(part_pricing, 'NetCorePrice')
                        price = get_text(part_pricing, 'Price')
                        return_old_part = get_text(part_pricing, 'ReturnOldPart')
                        
                        # Extract Category > Name from PCDBPart
                        pcdb_part = next((e for e in part_pricing if strip_ns(e.tag) == 'PCDBPart'), None)
                        if pcdb_part is not None:
                            category_elem = next((e for e in pcdb_part if strip_ns(e.tag) == 'Category'), None)
                            if category_elem is not None:
                                category_name = get_text(category_elem, 'Name')
                            part_terminology_name = get_text(pcdb_part, 'PartTerminologyName')
        
        # Build details dict
        result['details'] = {
            'part_number': part_number,
            'country_name': country_name,
            'manufacturer_name': manufacturer_name,
            'effective_date': effective_date,
            'is_current': is_current,
            'oepr_part_number': oepr_part_number,
            'motor_part_number': motor_part_number,
            'net_core_price': net_core_price,
            'category_name': category_name,
            'part_terminology_name': part_terminology_name,
            'price': price,
            'return_old_part': return_old_part
        }
    
    return result

In [6]:
# # Test the function
# result = extract_keywords_from_xml(resp)

# print("Extracted Keywords:")
# print(f"  Status: {result['status']}")
# print(f"  Status Code: {result['status_code']}")
# print(f"  Application ID: {result['application_id']}")
# print(f"  All Application IDs: {result['application_ids']}")

# # Convert to JSON
# print("\nAs JSON:")
# print(json.dumps({
#     'status': result['status'],
#     'status_code': result['status_code'],
#     'application_id': result['application_id']
# }, indent=2))

<h2>Get Vehicle Info by VIN</h2>

<p>/v1/Information/Vehicles/Search/ByVIN?vin={VIN}</p>

In [7]:
vin = "1FTEW1E45KFB21693"

veh_vin = {
            "US":{""
                    "escape_2014": "3FA6P0HD1ER388009",
                    "escape_2020": "3FA6P0D9XLR115438"},
            "CA":{
                "escape_2014": "1FMCU9G97EUB92197",
                "escape_2025": "1FMCU9NZXSUA08739"}
            }

In [8]:
def step_1(Vin):
    resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={Vin}")
    data = parse_vehicle_search(resp)
    
    # IMPORTANT: Store the actual VIN input so it flows through the pipeline
    data['actual_vin'] = Vin

    print(f"Response Status: {data['status']} ({data['status_code']})\n")
    return data

<h2>Get Summary</h2>
<p>/v1/Information/Vehicles/Attributes/BaseVehicleId/{VehId}/Content/Summaries/Of/EstimatedWorkTimes</p>

In [9]:
def step2(data):

   vehicle_id = data["vehicles"][0]["base_vehicle_id"]
   system_id = "5"
   group_id = ""
   sub_group_id = ""

   resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}/Content/Summaries/Of/EstimatedWorkTimes/?systemID={system_id}&groupID={group_id}&subGroupID={sub_group_id}")

   if not resp:
      raise ValueError("API response fails - step 2")

   data_out = extract_keywords_from_xml(resp)
   data_out["vehicle_id"] = vehicle_id

   # NEW: Pass vehicle info through the pipeline
   vehicle_info = data["vehicles"][0]
   data_out["vin"] = data.get("actual_vin")  # Use actual VIN from step_1, not vehicle_id
   data_out["make_name"] = vehicle_info.get("make_name")
   data_out["model_name"] = vehicle_info.get("model_name")
   data_out["engine_description"] = vehicle_info.get("engine_description")
   data_out["year"] = vehicle_info.get("year")

   return data_out

In [10]:
def step3(data):
    """Extract ALL work items with VIN and vehicle info."""
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag

    def get_text(elem, tag_name):
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None

    vehicle_id = data["vehicle_id"]
    application_ids = data["application_ids"]

    # Extract vehicle info passed through from step2
    vin = data.get("vin")
    make_name = data.get("make_name")
    model_name = data.get("model_name")
    engine_description = data.get("engine_description")
    year = data.get("year")

    result_list = []

    for app in application_ids:
        app_id = app["id"]
        display_name = app["display_name"]

        resp = GetResponse(
            f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
            f"/Content/Details/Of/EstimatedWorkTimes/{app_id}"
        )

        root = ET.fromstring(resp)

        items_container = None
        for elem in root.iter():
            if strip_ns(elem.tag) == 'Items':
                items_container = elem
                break

        work_items_list = []

        if items_container is not None:
            for work_time_elem in items_container:
                if strip_ns(work_time_elem.tag) == 'EstimatedWorkTime':
                    required_skill = None
                    skill_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'RequiredSkill'), None)
                    if skill_elem is not None:
                        required_skill = get_text(skill_elem, 'Description')

                    job_description = None
                    notes_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'Notes'), None)
                    if notes_elem is not None:
                        note_elem = next((e for e in notes_elem if strip_ns(e.tag) == 'Note'), None)
                        if note_elem is not None:
                            job_description = get_text(note_elem, 'Text')

                    work_item = {
                        'vin': vin,
                        'make_name': make_name,
                        'model_name': model_name,
                        'engine_description': engine_description,
                        'year': year,
                        'application_id': app_id,
                        'vehicle_id': vehicle_id,
                        'job_description': job_description,
                        'additional_labor_time': get_text(work_time_elem, 'AdditionalLaborTime'),
                        'additional_labor_time_description': get_text(work_time_elem, 'AdditionalLaborTimeDescription'),
                        'additional_warranty_labor_time': get_text(work_time_elem, 'AdditionalWarrantyLaborTime'),
                        'all_labor_time': get_text(work_time_elem, 'AllLaborTime'),
                        'all_labor_time_description': get_text(work_time_elem, 'AllLaborTimeDescription'),
                        'all_warranty_labor_time': get_text(work_time_elem, 'AllWarrantyLaborTime'),
                        'base_labor_time': get_text(work_time_elem, 'BaseLaborTime'),
                        'base_labor_time_description': get_text(work_time_elem, 'BaseLaborTimeDescription'),
                        'base_warranty_labor_time': get_text(work_time_elem, 'BaseWarrantyLaborTime'),
                        'estimated_work_time_id': get_text(work_time_elem, 'EstimatedWorkTimeID'),
                        'labor_time_interval': get_text(work_time_elem, 'LaborTimeInterval'),
                        'required_skill': required_skill,
                        'service_type': get_text(work_time_elem, 'ServiceType'),
                        'base_labor_time_average': get_text(work_time_elem, 'BaseLaborTimeAverage'),
                        'is_active': get_text(work_time_elem, 'IsActive'),
                        'type': get_text(work_time_elem, 'Type')
                    }

                    work_items_list.append(work_item)

        service_dict = {display_name: work_items_list}
        result_list.append(service_dict)

    return result_list

In [11]:
def step_4(labor_items):
    """
    Enrich each work item with its parts data.
    Input: list of dicts where each dict is {service_name: [work_items]}
    Output: single dict with service_name: [work_items_with_parts]
    
    Parts are stored as a dict keyed by part_app_id (not a list).
    
    Flattened structure:
    {
        'Rack & Pinion Assembly R&R': [
            {
                'application_id': '244560030',
                ...labor fields...,
                'parts': {
                    'part_app_id_1': {
                        'part_number': 'KG9Z 3504-H',
                        'price': '2718.18',
                        ...
                    },
                    'part_app_id_2': {...}
                }
            }
        ],
        'Steering Knuckle R&R': [...]
    }
    """
    enriched_dict = {}
    
    # Flatten the list of dicts into a single dict
    for service_dict in labor_items:
        for display_name, work_items in service_dict.items():
            enriched_work_items = []
            
            for work_item in work_items:
                application_id = work_item['application_id']
                vehicle_id = work_item['vehicle_id']
                
                # Get parts summary for this work-time
                resp = GetResponse(
                    f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                    f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
                )
                
                # Extract all part application IDs
                part_app_ids = extract_all_part_application_ids(resp)
                
                # Get details for each part - store as dict keyed by part_app_id
                parts_dict = {}
                for part_app_id in part_app_ids:
                    resp = GetResponse(
                        f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                        f"/Content/Details/Of/Parts/{part_app_id}"
                    )
                    part_details = extract_part_details(resp)
                    
                    part_info = part_details.get('details', {})
                    parts_dict[part_app_id] = part_info
                
                enriched_work_item = {
                    **work_item,
                    'parts': parts_dict
                }
                
                enriched_work_items.append(enriched_work_item)
            
            enriched_dict[display_name] = enriched_work_items
    
    return enriched_dict

In [12]:
"""
UPDATED step_5: Returns a LIST of part details (not a single dict)
This is critical - step_5 MUST return a list for the pipeline to work.
"""
def step_5(data):
    """
    Get details for ALL parts (loop through all part_app_ids).
    IMPORTANT: Returns a LIST of dicts, NOT a single dict.
    """
    vehicle_id = data.get("vehicle_id")
    part_app_ids = data.get("part_app_ids", [])
    
    # Initialize result as a LIST
    all_parts_list = []
    
    # Loop through each part ID and get its details
    for part_app_id in part_app_ids:
        try:
            resp = GetResponse(
                f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                f"/Content/Details/Of/Parts/{part_app_id}"
            )

            part_details = extract_part_details(resp)
            
            # Append to list
            all_parts_list.append({
                'part_app_id': part_app_id,
                'part_data': part_details
            })
        except Exception as e:
            print(f"    Error getting part {part_app_id}: {e}")
    
    # CRITICAL: Return the list, not a single dict
    return all_parts_list



In [13]:
# veh_data = step_1(veh_vin["US"]["escape_2020"])



In [14]:

# veh_data
 

In [15]:

# labour_summary_items = step2(veh_data)
 

In [16]:
# labour_summary_items

In [17]:
# labour_items = step3(labour_summary_items)

In [18]:
# len(labour_items[4])

In [19]:
# resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{85215}/Content/Details/Of/EstimatedWorkTimes/{244560030}")


In [20]:

# print(resp)
 

In [21]:
# resp_4 = step_4(labour_details)

In [22]:
# resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{85215}/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{244579908}")
 

In [23]:
# print(resp)

In [24]:


# step_5(labour_more_details)

In [25]:
# labour_details["application_id"]

In [26]:
# labour_more_details = step_4(labour_details)

In [27]:
# step_5(labour_more_details)

In [28]:
# def run_pipeline(vin: str) -> List[Dict]:
#     """
#     Full pipeline function: takes a VIN and returns all parts information 
#     for every work-time application and related part.
    
#     Returns a list of dicts, one entry per (work-time × part) combination.
#     Each entry contains work-time metadata, labor details, and part details.
#     """
#     # Step 1: VIN lookup
#     resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")
#     vehicle_data = parse_vehicle_search(resp)
#     base_vehicle_id = vehicle_data["vehicles"][0]["base_vehicle_id"]
    
#     # Step 2: Get all work-time application IDs
#     resp = GetResponse(
#         f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}"
#         f"/Content/Summaries/Of/EstimatedWorkTimes/?systemID=5&groupID=&subGroupID="
#     )
#     summaries = extract_keywords_from_xml(resp)
    
#     results = []
    
#     # Step 3: Loop over each work-time application
#     for app in summaries["application_ids"]:
#         app_id = app["id"]
#         display_name = app["display_name"]
        
#         # Get labor details
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Details/Of/EstimatedWorkTimes/{app_id}"
#         )
#         labor = extract_estimated_work_time(resp)
        
#         # Get related part application IDs
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{app_id}"
#         )
#         part_app_ids = extract_all_part_application_ids(resp)
        
#         # Get details for each part
#         for part_app_id in part_app_ids:
#             resp = GetResponse(
#                 f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Details/Of/Parts/{part_app_id}"
#             )
#             part = extract_part_details(resp)
            
#             results.append({
#                 'work_time_application_id': app_id,
#                 'work_time_display_name': display_name,
#                 'labor_details': labor.get('details', {}),
#                 'part_details': part.get('details', {})
#             })
    
#     return results
# results = run_pipeline(veh_vin["US"]["escape_2020"])
# results

In [29]:
# def run_pipeline_v2(vin: str) -> Dict:
#     """
#     Pipeline that follows the exact same strategy as step1→step2→step3→step4→step5.
#     Loops through all application IDs and all parts, stores results by vehicle_id.
#     Uses modified step_4 and step_5 that handle lists of part IDs.
#     """
#     print(f"Starting pipeline for VIN: {vin}\n")
    
#     # STEP 1: Vehicle VIN lookup
#     print("STEP 1: Vehicle lookup...")
#     resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")
#     data = parse_vehicle_search(resp)
#     print(f"✓ Vehicle found\n")
    
#     # STEP 2: Get all work-time application IDs
#     print("STEP 2: Get work-time summaries...")
#     vehicle_id = data["vehicles"][0]["base_vehicle_id"]
#     system_id = "5"
#     resp = GetResponse(
#         f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#         f"/Content/Summaries/Of/EstimatedWorkTimes/?systemID={system_id}&groupID=&subGroupID="
#     )
#     app_ID = extract_keywords_from_xml(resp)
#     app_ID["vehicle_id"] = vehicle_id
#     print(f"✓ Found {len(app_ID['application_ids'])} work-time applications\n")
    
#     # Storage for all results organized by vehicle_id
#     all_results = {
#         vehicle_id: {
#             'vehicle_info': data["vehicles"][0],
#             'parts_data': []
#         }
#     }
    
#     # STEP 3-5: Loop through EACH application_id and ALL its parts
#     print("STEPS 3-5: Processing each application and its parts...\n")
#     for app_idx, app in enumerate(app_ID['application_ids'], 1):
#         application_id = app['id']
#         display_name = app['display_name']
        
#         print(f"  {app_idx}. Processing: {display_name} (ID: {application_id})")
        
#         # STEP 3: Get labor details
#         labour_details_data = {
#             'vehicle_id': vehicle_id,
#             'application_id': application_id,
#             'display_name': display_name
#         }
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#             f"/Content/Details/Of/EstimatedWorkTimes/{application_id}"
#         )
#         labour_data = extract_estimated_work_time(resp)
#         labour_details_data.update(labour_data)
#         print(f"     ✓ Got labor details")
        
#         # STEP 4: Get parts summary (get ALL part application IDs)
#         step4_result = step_4(labour_details_data)
#         part_app_ids = step4_result.get("part_app_ids", [])
#         print(f"     ✓ Found {len(part_app_ids)} parts")
        
#         # STEP 5: Loop through EACH part and get details
#         parts_list = step_5(step4_result)
        
#         # Ensure parts_list is actually a list
#         if not isinstance(parts_list, list):
#             print(f"     ⚠️  Step 5 returned {type(parts_list)}, expected list")
#             parts_list = []
        
#         for part_idx, part_result in enumerate(parts_list, 1):
#             # Handle both old and new data structures
#             if isinstance(part_result, dict):
#                 part_number = part_result.get('part_data', {}).get('details', {}).get('part_number', 'N/A')
#                 part_app_id = part_result.get('part_app_id')
#                 part_details = part_result.get('part_data', {}).get('details', {})
#             else:
#                 part_number = 'N/A'
#                 part_app_id = 'N/A'
#                 part_details = {}
            
#             # Store combined result
#             result_entry = {
#                 'work_time': {
#                     'application_id': application_id,
#                     'display_name': display_name,
#                     'labor_details': labour_data.get('details', {})
#                 },
#                 'part': {
#                     'application_id': part_app_id,
#                     'details': part_details
#                 }
#             }
#             all_results[vehicle_id]['parts_data'].append(result_entry)
#             print(f"       {part_idx}. ✓ {part_number}")
        
#         print()
    
#     total_parts = len(all_results[vehicle_id]['parts_data'])
#     print(f"\n✓ Pipeline complete!")
#     print(f"  Vehicle ID: {vehicle_id}")
#     print(f"  Total work-times: {len(app_ID['application_ids'])}")
#     print(f"  Total parts: {total_parts}\n")
    
#     return all_results

# # Test with the working VIN
# results_v2 = run_pipeline_v2(veh_vin["US"]["escape_2020"])
# print(f"\nFinal result structure: {list(results_v2.keys())}")

In [30]:
# results_v2["85215"]["parts_data"]

In [31]:
# # CRITICAL DEBUG: Understand why extract_all_part_application_ids fails
# print("="*80)
# print("DEBUGGING: XML Element Structure Analysis")
# print("="*80 + "\n")

# vehicle_id = "85215"
# application_id = "244560030"

# resp = GetResponse(
#     f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#     f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
# )

# root = ET.fromstring(resp)

# def strip_ns(tag):
#     return tag.split('}')[-1] if '}' in tag else tag

# # Step 1: Find all PartApplicationSummary elements
# print("Step 1: Looking for PartApplicationSummary elements...\n")
# part_app_summaries = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary':
#         part_app_summaries.append(elem)
#         print(f"✓ Found PartApplicationSummary")
#         print(f"  Full tag (with namespace): '{elem.tag}'")
#         print(f"  Stripped tag: '{tag}'")
        
#         # Check its children
#         print(f"  Direct children:")
#         child_count = 0
#         for child in elem:
#             child_count += 1
#             child_tag = strip_ns(child.tag)
#             print(f"    [{child_count}] {child_tag:30s} = {child.text[:50] if child.text else 'None'}")
            
#             # If it's ApplicationID, show it
#             if child_tag == 'ApplicationID':
#                 print(f"        → FOUND ApplicationID: {child.text}")
        
#         if child_count == 0:
#             print(f"    (no direct children)")
#         print()

# print(f"Total PartApplicationSummary elements found: {len(part_app_summaries)}\n")

# # Step 2: Try the extraction with detailed steps
# print("Step 2: Simulating extract_all_part_application_ids logic...\n")
# ids = []
# containers_found = 0
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary' or tag == 'PartApp':
#         containers_found += 1
#         print(f"Found container: {tag}")
        
#         # Look for ApplicationID child
#         for e in elem:
#             e_tag = strip_ns(e.tag)
#             if e_tag == 'ApplicationID':
#                 print(f"  ✓ Found child ApplicationID: {e.text}")
#                 if e.text:
#                     ids.append(e.text)
        
#         # Also try with next() like in the function
#         app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
#         if app_id_elem is not None:
#             print(f"  next() also found: {app_id_elem.text}")
#         else:
#             print(f"  next() returned None")

# print(f"\nTotal containers found: {containers_found}")
# print(f"Total IDs extracted: {ids}\n")

# # Step 3: Show what extract_application_id would find
# print("Step 3: What extract_application_id finds...\n")
# app_ids = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'ApplicationID' and elem.text:
#         app_ids.append(elem.text)
#         print(f"Found ApplicationID: {elem.text}")

# print(f"Total ApplicationID elements: {len(app_ids)}\n")

<h2>Get Details</h2>
<p>/Vehicles/Attributes/BaseVehicleId/{VehicleId}/Content/Details/Of/EstimatedWorkTimes/{ApplicationId}</p>

<h2>Whatever link you want</h2>

In [32]:
# resp = GetResponse("/v1/Information/Vehicles/Attributes/BaseVehicleId/81898/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/185575044")
# print(resp)

In [33]:
# # CRITICAL DEBUG: Understand why extract_all_part_application_ids fails
# print("="*80)
# print("DEBUGGING: XML Element Structure Analysis")
# print("="*80 + "\n")

# vehicle_id = "85215"
# application_id = "244560030"

# resp = GetResponse(
#     f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#     f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
# )

# root = ET.fromstring(resp)

# def strip_ns(tag):
#     return tag.split('}')[-1] if '}' in tag else tag

# # Step 1: Find all PartApplicationSummary elements
# print("Step 1: Looking for PartApplicationSummary elements...\n")
# part_app_summaries = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary':
#         part_app_summaries.append(elem)
#         print(f"✓ Found PartApplicationSummary")
#         print(f"  Full tag (with namespace): '{elem.tag}'")
#         print(f"  Stripped tag: '{tag}'")
        
#         # Check its children
#         print(f"  Direct children:")
#         child_count = 0
#         for child in elem:
#             child_count += 1
#             child_tag = strip_ns(child.tag)
#             print(f"    [{child_count}] {child_tag:30s} = {child.text[:50] if child.text else 'None'}")
            
#             # If it's ApplicationID, show it
#             if child_tag == 'ApplicationID':
#                 print(f"        → FOUND ApplicationID: {child.text}")
        
#         if child_count == 0:
#             print(f"    (no direct children)")
#         print()

# print(f"Total PartApplicationSummary elements found: {len(part_app_summaries)}\n")

# # Step 2: Try the extraction with detailed steps
# print("Step 2: Simulating extract_all_part_application_ids logic...\n")
# ids = []
# containers_found = 0
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary' or tag == 'PartApp':
#         containers_found += 1
#         print(f"Found container: {tag}")
        
#         # Look for ApplicationID child
#         for e in elem:
#             e_tag = strip_ns(e.tag)
#             if e_tag == 'ApplicationID':
#                 print(f"  ✓ Found child ApplicationID: {e.text}")
#                 if e.text:
#                     ids.append(e.text)
        
#         # Also try with next() like in the function
#         app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
#         if app_id_elem is not None:
#             print(f"  next() also found: {app_id_elem.text}")
#         else:
#             print(f"  next() returned None")

# print(f"\nTotal containers found: {containers_found}")
# print(f"Total IDs extracted: {ids}\n")

# # Step 3: Show what extract_application_id would find
# print("Step 3: What extract_application_id finds...\n")
# app_ids = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'ApplicationID' and elem.text:
#         app_ids.append(elem.text)
#         print(f"Found ApplicationID: {elem.text}")

# print(f"Total ApplicationID elements: {len(app_ids)}\n")

In [34]:
def flatten_vin_dict(vin_dict: Dict) -> List[str]:
    """
    Extract all VINs from a nested dictionary structure.
    
    Example input:
    {
        "US": {
            "escape_2014": "3FA6P0HD1ER388009",
            "escape_2020": "3FA6P0D9XLR115438"
        },
        "CA": {
            "escape_2014": "1FMCU9G97EUB92197"
        }
    }
    
    Returns: ["3FA6P0HD1ER388009", "3FA6P0D9XLR115438", "1FMCU9G97EUB92197"]
    """
    vins = []
    for country, vehicles in vin_dict.items():
        if isinstance(vehicles, dict):
            for vehicle_name, vin in vehicles.items():
                vins.append(vin)
    return vins


def run_pipeline(vins):
    """
    Complete pipeline: Supports single VIN, list of VINs, or veh_vin dict.

    Args:
        vins: Can be:
          - Single VIN string: "3FA6P0D9XLR115438"
          - List of VIN strings: ["3FA6P0D9XLR115438", "1FMCU9G97EUB92197"]
          - veh_vin dict: {"US": {"escape_2020": "3FA6P0D9XLR115438", ...}, ...}

    Returns:
        Dict with all services/work items/parts from all VINs
    """
    # Handle all input types: string, list, or dict
    if isinstance(vins, str):
        vins = [vins]
    elif isinstance(vins, dict):
        vins = flatten_vin_dict(vins)

    print(f"Starting pipeline for {len(vins)} VIN(s)\n")

    all_results = {}

    for vin in vins:
        print(f"Processing VIN: {vin}")
        print("-" * 80)

        print("STEP 1: Vehicle lookup...")
        vehicle_data = step_1(vin)
        vin_info = vehicle_data['vehicles'][0]
        print(f"OK: {vin_info['make_name']} {vin_info['model_name']} {vin_info['year']}")

        print("STEP 2: Get work-time summaries...")
        summaries = step2(vehicle_data)
        num_apps = len(summaries['application_ids'])
        print(f"OK: Found {num_apps} work-time applications")

        print("STEP 3: Extract work items...")
        labor_items = step3(summaries)
        print(f"OK: Processed {len(labor_items)} services")

        print("STEP 4: Enrich with parts data...")
        complete_data = step_4(labor_items)

        total_parts = sum(len(work_item.get('parts', {}))
                         for work_items in complete_data.values()
                         for work_item in work_items)
        print(f"OK: Found {total_parts} total parts\n")

        # Merge results
        for service_name, work_items in complete_data.items():
            if service_name in all_results:
                all_results[service_name].extend(work_items)
            else:
                all_results[service_name] = work_items

    print("=" * 80)
    print(f"Pipeline complete!")
    print(f"VINs processed: {len(vins)}")
    print(f"Services: {len(all_results)}")
    total_all = sum(len(items) for items in all_results.values())
    print(f"Total work items: {total_all}\n")

    return all_results

In [35]:

# pipe_1 = run_pipeline(veh_vin["US"]["escape_2020"]) 
 

In [36]:
# pipe_1["Rack & Pinion Assembly R&R"][0]["parts"]

In [37]:
# Import Excel export functions from pipeline_to_excel module
from pipeline_to_excel import pipeline_to_excel, display_summary
import pandas as pd

print("Excel export functions imported successfully!")


def search_labour_and_parts(year: int, make: str = None, excel_file: str = "./output/motor_data.xlsx"):
    """
    Search labour and parts data by year and optional make.
    Only displays items with complete data (skips empty records).
    
    Relationships:
    - Service → Labour Item (with job description) → Parts (with part number)
    
    Args:
        year (int): Vehicle year (required) - e.g., 2020
        make (str): Vehicle make (optional) - e.g., 'Ford', 'Honda'
        excel_file (str): Path to Excel file (default: "./output/motor_data.xlsx")
    
    Usage:
        search_labour_and_parts(2020)  # All 2020 vehicles
        search_labour_and_parts(2020, make="Ford")  # Only 2020 Ford vehicles
    """
    # Load the Excel file
    df = pd.read_excel(excel_file)
    
    # Filter by year (required)
    filtered_df = df[df['year'] == year].copy()
    
    if filtered_df.empty:
        print(f"❌ No data found for year {year}")
        return
    
    # Filter by make if provided
    if make:
        filtered_df = filtered_df[filtered_df['make_name'].str.lower() == make.lower()]
        if filtered_df.empty:
            print(f"❌ No data found for {make} in year {year}")
            return
        print(f"\n🔍 Searching: {make.title()} vehicles from {year}\n")
    else:
        print(f"\n🔍 Searching: All vehicles from {year}\n")
    
    # Get unique services
    services = filtered_df['service_name'].unique()
    
    print("=" * 110)
    print(f"FOUND {len(services)} SERVICE(S) across {filtered_df['vin'].nunique()} VEHICLE(S)")
    print("=" * 110 + "\n")
    
    # Process each service
    for service_idx, service_name in enumerate(services, 1):
        service_data = filtered_df[filtered_df['service_name'] == service_name]
        
        # Get labour records with actual job descriptions (filter out empty ones)
        labour_records = service_data[
            (service_data['record_type'] == 'labour') & 
            (pd.notna(service_data['job_description'])) &
            (service_data['job_description'].str.strip() != '')
        ].drop_duplicates(subset=['application_id'])
        
        if labour_records.empty:
            continue
        
        print(f"\n[{service_idx}] 📋 SERVICE: {service_name}")
        print("─" * 110)
        
        # Process each labour item
        for labour_idx, (idx, labour) in enumerate(labour_records.iterrows(), 1):
            app_id = labour['application_id']
            
            print(f"\n   ├─ LABOUR #{labour_idx}")
            print(f"   │  ├─ Job: {labour['job_description']}")
            
            # Only show labour details if they have actual values
            if pd.notna(labour['base_labor_time']) and labour['base_labor_time'] != '' and labour['base_labor_time'] != '0':
                print(f"   │  ├─ Base Time: {labour['base_labor_time']} hrs")
            if pd.notna(labour['all_labor_time']) and labour['all_labor_time'] != '' and labour['all_labor_time'] != '0':
                print(f"   │  ├─ Total Time: {labour['all_labor_time']} hrs")
            if pd.notna(labour['required_skill']) and labour['required_skill'].strip() != '':
                print(f"   │  ├─ Skill: {labour['required_skill']}")
            if pd.notna(labour['service_type']) and labour['service_type'].strip() != '':
                print(f"   │  └─ Type: {labour['service_type']}")
            
            # Get parts for THIS labour with valid part numbers
            parts_for_labour = service_data[
                (service_data['record_type'] == 'part') & 
                (service_data['application_id'] == app_id) &
                (pd.notna(service_data['oepr_part_number'])) &
                (service_data['oepr_part_number'].str.strip() != '') &
                (service_data['oepr_part_number'] != 'NR') &
                (service_data['oepr_part_number'] != 'nan')
            ].drop_duplicates(subset=['part_app_id'])
            
            if not parts_for_labour.empty:
                print(f"   │")
                print(f"   │  📦 PARTS FOR THIS LABOUR ({len(parts_for_labour)} parts):")
                
                for part_idx, (p_idx, part) in enumerate(parts_for_labour.iterrows(), 1):
                    is_last = (part_idx == len(parts_for_labour))
                    prefix = "   │     └─" if is_last else "   │     ├─"
                    next_prefix = "   │        " if is_last else "   │     │  "
                    
                    print(f"{prefix} Part #{part_idx}: {part['oepr_part_number']}")
                    
                    # Build part details list - only show if data exists
                    details = []
                    if pd.notna(part['part_terminology_name']) and part['part_terminology_name'].strip() != '':
                        details.append(f"├─ Name: {part['part_terminology_name']}")
                    if pd.notna(part['category_name']) and part['category_name'].strip() != '':
                        details.append(f"├─ Category: {part['category_name']}")
                    if pd.notna(part['manufacturer_name']) and part['manufacturer_name'].strip() != '':
                        details.append(f"├─ Manufacturer: {part['manufacturer_name']}")
                    if pd.notna(part['price']) and str(part['price']).strip() != '' and str(part['price']) != '0':
                        details.append(f"└─ Price: ${part['price']}")
                    
                    # Print only the details that have data
                    for detail_idx, detail in enumerate(details):
                        print(f"{next_prefix}{detail}")
    
    # Summary statistics - only count records with actual data
    labour_with_data = filtered_df[
        (filtered_df['record_type'] == 'labour') & 
        (pd.notna(filtered_df['job_description'])) &
        (filtered_df['job_description'].str.strip() != '')
    ].shape[0]
    
    parts_with_data = filtered_df[
        (filtered_df['record_type'] == 'part') & 
        (pd.notna(filtered_df['oepr_part_number'])) &
        (filtered_df['oepr_part_number'].str.strip() != '') &
        (filtered_df['oepr_part_number'] != 'NR') &
        (filtered_df['oepr_part_number'] != 'nan')
    ].shape[0]
    
    print("\n" + "=" * 110)
    print(f"SUMMARY:")
    print(f"  • Years: {filtered_df['year'].unique().tolist()}")
    print(f"  • Makes: {filtered_df['make_name'].unique().tolist()}")
    print(f"  • Vehicles: {filtered_df['vin'].nunique()} unique VIN(s)")
    print(f"  • Services: {filtered_df['service_name'].nunique()}")
    print(f"  • Labour Items (with data): {labour_with_data}")
    print(f"  • Parts (with data): {parts_with_data}")
    print("=" * 110 + "\n")


def compare_years(make: str, year1: int, year2: int, excel_file: str = "./output/motor_data.xlsx"):
    """
    Compare labour and parts data between two years for the same make.
    Shows side-by-side comparison with summary statistics and detailed analysis.
    
    Args:
        make (str): Vehicle make - e.g., 'Ford', 'Honda'
        year1 (int): First year to compare - e.g., 2020
        year2 (int): Second year to compare - e.g., 2014
        excel_file (str): Path to Excel file (default: "./output/motor_data.xlsx")
    
    Usage:
        compare_years("Ford", 2020, 2014)
    """
    # Load the Excel file
    df = pd.read_excel(excel_file)
    
    # Filter by make
    df_make = df[df['make_name'].str.lower() == make.lower()].copy()
    
    if df_make.empty:
        print(f"❌ No data found for {make}")
        return
    
    # Filter for both years
    df_year1 = df_make[df_make['year'] == year1].copy()
    df_year2 = df_make[df_make['year'] == year2].copy()
    
    if df_year1.empty:
        print(f"❌ No data found for {make} in year {year1}")
        return
    if df_year2.empty:
        print(f"❌ No data found for {make} in year {year2}")
        return
    
    print(f"\n{'='*130}")
    print(f"COMPARISON: {make.upper()} | {year1} vs {year2}".center(130))
    print(f"{'='*130}\n")
    
    # ============ SUMMARY TABLE ============
    print("📊 SUMMARY STATISTICS\n")
    
    # Calculate stats for year1
    services_y1 = df_year1['service_name'].nunique()
    labour_y1 = df_year1[
        (df_year1['record_type'] == 'labour') & 
        (pd.notna(df_year1['job_description'])) &
        (df_year1['job_description'].str.strip() != '')
    ]['application_id'].nunique()
    parts_y1 = df_year1[
        (df_year1['record_type'] == 'part') & 
        (pd.notna(df_year1['oepr_part_number'])) &
        (df_year1['oepr_part_number'].str.strip() != '') &
        (df_year1['oepr_part_number'] != 'NR')
    ]['oepr_part_number'].nunique()
    
    # Calculate stats for year2
    services_y2 = df_year2['service_name'].nunique()
    labour_y2 = df_year2[
        (df_year2['record_type'] == 'labour') & 
        (pd.notna(df_year2['job_description'])) &
        (df_year2['job_description'].str.strip() != '')
    ]['application_id'].nunique()
    parts_y2 = df_year2[
        (df_year2['record_type'] == 'part') & 
        (pd.notna(df_year2['oepr_part_number'])) &
        (df_year2['oepr_part_number'].str.strip() != '') &
        (df_year2['oepr_part_number'] != 'NR')
    ]['oepr_part_number'].nunique()
    
    # Create summary dataframe
    summary_data = {
        'Metric': ['Services', 'Labour Items', 'Unique Parts'],
        year1: [services_y1, labour_y1, parts_y1],
        year2: [services_y2, labour_y2, parts_y2],
        'Difference': [services_y2 - services_y1, labour_y2 - labour_y1, parts_y2 - parts_y1]
    }
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    # ============ SERVICE COMPARISON ============
    print(f"\n\n🔧 LABOUR & PARTS COMPARISON BY SERVICE\n")
    
    # Get services for both years
    services_y1_list = set(df_year1['service_name'].unique())
    services_y2_list = set(df_year2['service_name'].unique())
    
    common_services = services_y1_list & services_y2_list
    unique_y1 = services_y1_list - services_y2_list
    unique_y2 = services_y2_list - services_y1_list
    
    # Process common services
    if common_services:
        print("─" * 130)
        print(f"COMMON SERVICES ({len(common_services)})".ljust(130, "─"))
        print("─" * 130)
        
        for service_name in sorted(common_services):
            service_y1 = df_year1[df_year1['service_name'] == service_name]
            service_y2 = df_year2[df_year2['service_name'] == service_name]
            
            # Get parts for this service
            parts_y1_set = set(service_y1[
                (service_y1['record_type'] == 'part') &
                (pd.notna(service_y1['oepr_part_number'])) &
                (service_y1['oepr_part_number'].str.strip() != '') &
                (service_y1['oepr_part_number'] != 'NR')
            ]['oepr_part_number'].unique())
            
            parts_y2_set = set(service_y2[
                (service_y2['record_type'] == 'part') &
                (pd.notna(service_y2['oepr_part_number'])) &
                (service_y2['oepr_part_number'].str.strip() != '') &
                (service_y2['oepr_part_number'] != 'NR')
            ]['oepr_part_number'].unique())
            
            common_parts = parts_y1_set & parts_y2_set
            unique_parts_y1 = parts_y1_set - parts_y2_set
            unique_parts_y2 = parts_y2_set - parts_y1_set
            
            print(f"\n📋 {service_name}")
            print(f"   {year1}: {len(parts_y1_set)} parts  |  {year2}: {len(parts_y2_set)} parts")
            print(f"   Common: {len(common_parts)}  |  {year1} Only: {len(unique_parts_y1)}  |  {year2} Only: {len(unique_parts_y2)}")
            
            if unique_parts_y1:
                print(f"   → {year1} Only: {', '.join(sorted(list(unique_parts_y1)[:5]))}")
                if len(unique_parts_y1) > 5:
                    print(f"      ... and {len(unique_parts_y1) - 5} more")
            
            if unique_parts_y2:
                print(f"   → {year2} Only: {', '.join(sorted(list(unique_parts_y2)[:5]))}")
                if len(unique_parts_y2) > 5:
                    print(f"      ... and {len(unique_parts_y2) - 5} more")
    
    # Services unique to year1
    if unique_y1:
        print(f"\n{'─' * 130}")
        print(f"SERVICES ONLY IN {year1} ({len(unique_y1)})".ljust(130, "─"))
        print(f"{'─' * 130}")
        for service in sorted(unique_y1):
            parts_count = df_year1[
                (df_year1['service_name'] == service) &
                (df_year1['record_type'] == 'part') &
                (pd.notna(df_year1['oepr_part_number'])) &
                (df_year1['oepr_part_number'].str.strip() != '') &
                (df_year1['oepr_part_number'] != 'NR')
            ]['oepr_part_number'].nunique()
            print(f"   • {service} ({parts_count} parts)")
    
    # Services unique to year2
    if unique_y2:
        print(f"\n{'─' * 130}")
        print(f"SERVICES ONLY IN {year2} ({len(unique_y2)})".ljust(130, "─"))
        print(f"{'─' * 130}")
        for service in sorted(unique_y2):
            parts_count = df_year2[
                (df_year2['service_name'] == service) &
                (df_year2['record_type'] == 'part') &
                (pd.notna(df_year2['oepr_part_number'])) &
                (df_year2['oepr_part_number'].str.strip() != '') &
                (df_year2['oepr_part_number'] != 'NR')
            ]['oepr_part_number'].nunique()
            print(f"   • {service} ({parts_count} parts)")
    
    print(f"\n{'='*130}\n")

Excel export functions imported successfully!


In [38]:
from comparison_functions import compare_years

In [39]:
# # STEP 1: Run the complete pipeline
# results = run_pipeline(veh_vin)

# # STEP 2: Convert to Excel and save with deduplication
# file_path = pipeline_to_excel(results)



In [40]:

import pandas as pd
 

In [41]:

data = pd.read_excel("./output/motor_data.xlsx")
 

In [42]:
# data["country_name"].unique()
    
data[data["country_name"].str.contains("Canada", case=False, na=False)]["model_name"].unique()

array(['Fusion', 'Escape'], dtype=object)

In [43]:

veh_vin
 

{'US': {'escape_2014': '3FA6P0HD1ER388009',
  'escape_2020': '3FA6P0D9XLR115438'},
 'CA': {'escape_2014': '1FMCU9G97EUB92197',
  'escape_2025': '1FMCU9NZXSUA08739'}}

In [44]:

# Compare Ford Escape across years

compare_years("Ford", "Escape", 2025, 2014, country="United States")
 


                                                             COMPARISON: FORD ESCAPE | 2025 vs 2014                                                             

📊 SUMMARY STATISTICS

      Metric  2025  2014  Δ (Difference)
    Services     3     3               0
Labour Items     3     3               0
Unique Parts    11     9              -2
🔍 DEBUG: Total labour records (2025): 0, (2014): 0


📋 DETAILED SERVICE HIERARCHY (SIDE-BY-SIDE)

                             2025 DETAILS                               │                               2014 DETAILS                             
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

🔧 Rack & Pinion Assembly R&R                                            │  
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    📦 LX

In [45]:
compare_years("Ford", "Fusion", 2020, 2014, country="Canada")


                                                             COMPARISON: FORD FUSION | 2020 vs 2014                                                             

📊 SUMMARY STATISTICS

      Metric  2020  2014  Δ (Difference)
    Services     2     3               1
Labour Items     2     3               1
Unique Parts     5     7               2
🔍 DEBUG: Total labour records (2020): 0, (2014): 0


📋 DETAILED SERVICE HIERARCHY (SIDE-BY-SIDE)

                             2020 DETAILS                               │                               2014 DETAILS                             
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

🔧 Steering Knuckle R&R                                                  │  
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    📦 DG